# 07 · Modelling pipeline: split, models, evaluation, cost curve, error types

> **Status (2026-11-25):** the conclusions and findings in this notebook were written on the **uncleaned** pairs (`pairs_scoped.parquet`). The code now runs on the cleaned pairs (`config.PAIRS_INPUT` = `pairs_clean.parquet`); after the re-run, the numbers in the conclusions must be updated.

Everything needed from features to results, in one notebook first. Once we are happy with it, the code moves to `src/screening/` and `scripts/04–06` (see the last section).

- **Input:** `config.PAIRS_INPUT` (the cleaned pairs, `pairs_clean.parquet`) and the two feature files: `features_name.parquet` (15 `n_` features) and `features_record.parquet` (4 `r_` features, missing fields handled as set in `config.R_MISSING_POLICY`). Re-run `make features` (scripts 03a + 03b) whenever `PAIRS_INPUT` or a feature changes.
- **Test set:** Vova's split (`pairs_split.parquet`, notebook 02) is used to keep his **test** pairs out: this notebook loads only `train` and `val`. All comparisons are repeated splits by person group inside these pairs, so the test set stays untouched until the final run.
- **Score:** false alarms at a fixed, very high recall (95 / 98 / 99 / 99.5%), not AUC.

Abbreviations: **FA rate** = false-alarm rate (share of negative pairs that get an alarm); **pp** = percentage points; **dev** = development pairs; **logreg** = logistic regression; **hgb** = scikit-learn HistGradientBoosting; **giant** = pairs in the largest connected component.

In [ ]:
import sys, time, hashlib
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from screening import config as C

pd.set_option("display.width", 160)
RECALLS = [0.95, 0.98, 0.99, 0.995]
TARGET = 0.99

## 1 · Load pairs and features

Features come from the two feature files (`features_name.parquet`, `features_record.parquet`), merged on `pair_id`. Both are required. Only pairs labelled `positive` or `negative` are used. `is_person` (both sides are persons) is added as a feature: it is the schema, not a leak.

In [ ]:
pairs = pd.read_parquet(C.PAIRS_INPUT, columns=["pair_id", "left_id", "right_id", "judgement", "left_schema",
                                                "left_name", "right_name"])
for path, script in [(C.FEATURES_NAME, "03a_name_features.py"), (C.FEATURES_RECORD, "03b_record_features.py")]:
    assert path.exists(), f"{path.name} missing: run scripts/{script} (or make features)"
feats = pd.read_parquet(C.FEATURES_NAME)
rec = pd.read_parquet(C.FEATURES_RECORD)
assert len(rec) == len(feats), "name and record features must come from the same PAIRS_INPUT"
feats = feats.merge(rec, on="pair_id", validate="one_to_one")
parts = ["name", "record"]
print(f"record features: {C.R_FEATURES}; missing policy '{C.R_MISSING_POLICY}'")
assert len(feats) == len(pairs), f"features {len(feats):,} rows vs pairs {len(pairs):,}: re-run the feature scripts"

df = pairs.merge(feats, on="pair_id", validate="one_to_one")
print("judgement values:", df["judgement"].value_counts().to_dict())
df = df[df["judgement"].isin(["positive", "negative"])].reset_index(drop=True)
df["y"] = (df["judgement"] == "positive").astype(int)
df["is_person"] = (df["left_schema"] == "Person").astype(int)

# keep Vova's test pairs out: only train and val are used in this notebook
vsplit = pd.read_parquet(C.PAIRS_SPLIT, columns=["pair_id", "split"])
df = df.merge(vsplit, on="pair_id", how="inner", validate="one_to_one")
print("Vova's split:", df["split"].value_counts().to_dict(), "(test is dropped below)")
df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"input {C.PAIRS_INPUT.name}: {len(df):,} labelled pairs, {df['y'].mean():.1%} positive; features: {' + '.join(parts)}")

## 2 · Split rules

Two rules, both based on a hash of an ID and a salt (same salt = same split, even after cleaning removes pairs; new salt = new split):

- **By person group (used everywhere in this notebook):** IDs linked by positive pairs form a group (one real entity, as the analysts decided it); a whole group goes to one side. This is the same rule as Vova's split (notebook 02). A pair is a **test** pair if both groups are on the test side, a **train** pair if both are on the train side, **cross** otherwise (dropped).
- **By entity ID (only for the comparison in section 10):** the same, but every ID is assigned on its own, so two IDs of one real entity can end up on different sides.

In [ ]:
def unit_hash(ids, salt):
    return np.array([int(hashlib.sha1(f"{salt}|{i}".encode()).hexdigest()[:12], 16) / 16**12 for i in ids])

def pair_side(d, share, salt, names=("train", "test")):
    ents = pd.unique(pd.concat([d["left_id"], d["right_id"]]))
    is_test = pd.Series(unit_hash(ents, salt) < share, index=ents)
    lt, rt = d["left_id"].map(is_test).to_numpy(), d["right_id"].map(is_test).to_numpy()
    return pd.Series(np.where(lt & rt, names[1], np.where(~lt & ~rt, names[0], "cross")), index=d.index)

# person groups: IDs linked by positive pairs; key = smallest ID of the group (stable)
_parent = {}
def _find(x):
    _parent.setdefault(x, x)
    while _parent[x] != x:
        _parent[x] = _parent[_parent[x]]
        x = _parent[x]
    return x
for a, b in zip(df.loc[df["y"] == 1, "left_id"], df.loc[df["y"] == 1, "right_id"]):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        _parent[ra] = rb
_ents = pd.unique(pd.concat([df["left_id"], df["right_id"]]))
_root = pd.Series([_find(e) for e in _ents], index=_ents)
group_key = pd.Series(_root.index, index=_root.index).groupby(_root.values).transform("min")
df["grp_l"], df["grp_r"] = df["left_id"].map(group_key), df["right_id"].map(group_key)

def group_side(d, share, salt):
    grps = pd.unique(pd.concat([d["grp_l"], d["grp_r"]]))
    is_test = pd.Series(unit_hash(grps, salt) < share, index=grps)
    lt, rt = d["grp_l"].map(is_test).to_numpy(), d["grp_r"].map(is_test).to_numpy()
    return pd.Series(np.where(lt & rt, "test", np.where(~lt & ~rt, "train", "cross")), index=d.index)

print(f"{len(_ents):,} IDs in {group_key.nunique():,} person groups")

## 3 · Giant component (for reporting only)

Union-find over all pairs; pairs in the largest connected component are labelled `giant`, all others `rest`. This label is **never a feature** (component size predicts the label, notebook 06). We use it only to report false alarms separately, because the giant component holds mostly negatives and the rest mostly positives.

In [ ]:
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a, b in zip(df["left_id"], df["right_id"]):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
comp = df["left_id"].map(find)
df["group"] = np.where(comp == comp.value_counts().index[0], "giant", "rest")
df.groupby("group").agg(pairs=("y", "size"), share_positive=("y", "mean")).round(3)

**Conclusion.**
- **The giant component** holds 252,590 pairs (58%) and only 39.2% of them are positive; the rest is 94.3% positive.
- **The two groups are different problems:** the giant component has the many, mostly easy negatives, the rest has few but hard ones.

**Next step:** always report false alarms for both groups (section 6).

**What the table shows:** pairs and share of positives in the giant component and in the rest.

**How to read it:** the giant component holds most pairs and a much lower share of positives than the rest.

## 4 · Evaluation helpers

- `at_recall(y, score, recall)`: the lowest score threshold that still finds the given share of positive pairs. All pairs with a score at or above it get an alarm (ties included, as in production). Returns the threshold, the number of false alarms, the FA rate and the number of alerts.
- `by_group(...)`: the threshold is chosen on **all** pairs; false alarms are then counted inside giant and rest separately.

In [ ]:
def at_recall(y, score, recall):
    y, score = np.asarray(y), np.asarray(score)
    n_pos, n_neg = int(y.sum()), int((y == 0).sum())
    order = np.argsort(-score, kind="stable")
    tp = np.cumsum(y[order])
    k = min(int(np.searchsorted(tp, np.ceil(recall * n_pos))), len(y) - 1)
    thr = score[order][k]
    alarm = score >= thr
    fa = int((alarm & (y == 0)).sum())
    return {"threshold": float(thr), "false_alarms": fa, "fa_rate": fa / n_neg, "alerts": int(alarm.sum()),
            "recall_reached": float((alarm & (y == 1)).sum() / n_pos)}

def by_group(y, score, groups, recalls=RECALLS):
    y, score, groups = np.asarray(y), np.asarray(score), np.asarray(groups)
    rows = []
    for r in recalls:
        alarm = score >= at_recall(y, score, r)["threshold"]
        for g in ["all", "giant", "rest"]:
            m = np.ones(len(y), bool) if g == "all" else groups == g
            neg = (y == 0) & m
            rows.append({"recall": r, "group": g, "fa_rate": (alarm & neg).sum() / max(neg.sum(), 1)})
    return pd.DataFrame(rows)

# quick self-check on a toy example: 4 positives, 4 negatives
yt = np.array([1, 1, 1, 1, 0, 0, 0, 0]); st = np.array([.9, .8, .7, .3, .6, .4, .2, .1])
print(at_recall(yt, st, 1.0))   # threshold 0.3 -> negatives 0.6 and 0.4 get an alarm: 2 false alarms

## 5 · Models

All four get the same feature table. Numeric features may be NaN, categorical features are pandas `category` columns.

| Model | Handling of NaN and categories | Role |
| --- | --- | --- |
| logreg | median imputation + missing indicators, scaling, one-hot categories | baseline |
| hgb | native | cross-check (used in notebooks 04/06) |
| lightgbm | native | main model |
| xgboost | native (`enable_categorical`) | main model |

`GRID` holds the parameter settings; setting 0 is the default, the others are tried in section 7.

In [ ]:
GRID = {
    "logreg":   [{"C": 1.0}],
    "hgb":      [{"max_iter": 300, "learning_rate": 0.1}],
    "lightgbm": [{"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 20},
                 {"n_estimators": 800, "learning_rate": 0.03, "num_leaves": 63, "min_child_samples": 20},
                 {"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 15, "min_child_samples": 50}],
    "xgboost":  [{"n_estimators": 400, "learning_rate": 0.05, "max_depth": 6},
                 {"n_estimators": 800, "learning_rate": 0.03, "max_depth": 8},
                 {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 4}],
}

def make_model(name, params, X):
    seed = C.RANDOM_SEED
    if name == "logreg":
        cat = [c for c in X.columns if str(X[c].dtype) == "category"]
        num = [c for c in X.columns if c not in cat]
        pre = ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
        return make_pipeline(pre, LogisticRegression(max_iter=2000, random_state=seed, **params))
    if name == "hgb":
        return HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=seed, **params)
    if name == "lightgbm":
        return LGBMClassifier(random_state=seed, verbose=-1, **params)
    if name == "xgboost":
        return XGBClassifier(tree_method="hist", enable_categorical=True, random_state=seed,
                             eval_metric="logloss", **params)
    raise ValueError(name)

FEATURE_SETS = {"name": ("n_",), "record": ("r_",), "name+record": ("n_", "r_")}
def feature_columns(set_name):
    cols = []
    for p in FEATURE_SETS[set_name]:
        part = [c for c in df.columns if c.startswith(p)]
        if not part:
            return None
        cols += part
    return cols + ["is_person"]

SETS = {s: feature_columns(s) for s in FEATURE_SETS if feature_columns(s) is not None}
print({s: len(c) for s, c in SETS.items()})

## 6 · Compare models

On Vova's train + val pairs: 5 repeated splits **by person group** (section 2, 30% of the groups to validation). Every model is trained on the inner training pairs and scored on the inner validation pairs. Default parameters (setting 0) here; tuning in section 7.

Takes a few minutes.

In [ ]:
dev = df   # development pairs = Vova's train + val (test excluded in section 1)
REPS = 5
inner = [(rep, group_side(dev, 0.3, f"group-{rep}")) for rep in range(REPS)]

def run(models_settings, sets=SETS):
    rows, scores = [], {}
    for rep, side in inner:
        tr, va = dev[side == "train"], dev[side == "test"]
        for s, cols in sets.items():
            for m, k in models_settings:
                t0 = time.time()
                model = make_model(m, GRID[m][k], tr[cols]).fit(tr[cols], tr["y"])
                s_va = model.predict_proba(va[cols])[:, 1]
                s_tr = model.predict_proba(tr[cols])[:, 1]
                scores[(rep, s, m, k)] = s_va
                g = by_group(va["y"], s_va, va["group"])
                for _, r in g.iterrows():
                    rows.append({"rep": rep, "set": s, "model": m, "setting": k, "recall": r["recall"],
                                 "group": r["group"], "fa_rate": r["fa_rate"],
                                 "train_fa": at_recall(tr["y"], s_tr, TARGET)["fa_rate"],
                                 "auc": roc_auc_score(va["y"], s_va), "seconds": time.time() - t0})
    return pd.DataFrame(rows), scores

t0 = time.time()
res, scores = run([(m, 0) for m in GRID])
print(f"{len(res['rep'].unique())} splits x {len(SETS)} feature sets x {len(GRID)} models in {time.time() - t0:.0f}s; "
      f"inner train ~{(inner[0][1] == 'train').sum():,}, validation ~{(inner[0][1] == 'test').sum():,} pairs")

### Table: false alarms at fixed recall

**What it shows:** FA rate in % on the inner validation pairs, mean ± standard deviation over the 5 splits, per feature set and model. AUC for comparison. Last column: seconds per fit.

**How to read it:** lower is better. Differences smaller than the standard deviation are noise. AUC will be close for all models; the columns at 99% and 99.5% recall show the differences that matter.

In [ ]:
def summary(res):
    a = res[res["group"] == "all"]
    t = a.groupby(["set", "model", "setting", "recall"])["fa_rate"].agg(["mean", "std"]) * 100
    t = t.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1).unstack("recall")
    t.columns = [f"FA % at {c:.1%}" for c in t.columns]
    extra = a.groupby(["set", "model", "setting"]).agg(auc=("auc", "mean"), seconds=("seconds", "mean"))
    return t.join(extra.round(4))
summary(res)

**Conclusion.**
- **The three tree models are equal:** LightGBM 2.91%, XGBoost 2.93% and HGB 2.95% false alarms at 99% recall. The differences are smaller than the standard deviation (0.10–0.15).
- **Logistic regression is clearly worse:** 5.29% at 99% and 10.05% at 99.5%.
- **AUC hides this:** 0.991 vs 0.994. The fixed-recall columns are the ones that matter.

**Next step:** check whether any tree model is better consistently (paired table below).

### Table: paired comparison at 99% recall

**What it shows:** for each model, in how many of the 5 splits it has fewer false alarms than each other model (same split, same feature set).

**How to read it:** 5 of 5 = consistently better; 3 of 5 or less = no reliable difference.

In [ ]:
def paired(res, set_name):
    a = res[(res["group"] == "all") & (res["recall"] == TARGET) & (res["set"] == set_name)]
    p = a.pivot_table(index="rep", columns=["model", "setting"], values="fa_rate")
    names = list(p.columns)
    return pd.DataFrame([[int((p[i] < p[j]).sum()) if i != j else None for j in names] for i in names],
                        index=[f"{m} ({k})" for m, k in names], columns=[f"vs {m} ({k})" for m, k in names])
for s in SETS:
    print(f"feature set: {s}  (row model better than column model in ... of {REPS} splits)")
    display(paired(res, s))

**Conclusion.**
- **No tree model is better in more than 3 of 5 splits,** so there is no reliable winner among them.
- **All tree models beat logistic regression in 5 of 5 splits.**

**Next step:** use LightGBM as main model (fastest of the three, about 16 s per fit) and XGBoost as a cross-check; logistic regression stays as baseline.

### Table: giant component vs the rest, and overfitting check (99% recall)

**What it shows:** FA rate in % counted inside the giant component and inside the rest (threshold chosen on all pairs), next to the overall rate. Then the FA rate on the inner training pairs vs the validation pairs.

**How to read it:** the overall rate is a mix of the two groups; the rest usually has far fewer negatives but harder ones. For overfitting, a gap of a few tenths of a percentage point is normal; a large gap means the model memorises training pairs.

In [ ]:
def groups_and_gap(res):
    t = res[res["recall"] == TARGET]
    g = t.groupby(["set", "model", "setting", "group"])["fa_rate"].mean().unstack("group") * 100
    a = t[t["group"] == "all"].groupby(["set", "model", "setting"]).agg(train=("train_fa", "mean"),
                                                                       valid=("fa_rate", "mean")) * 100
    a["valid − train (pp)"] = a["valid"] - a["train"]
    return g.join(a.add_prefix("gap: ")).round(2)
groups_and_gap(res)

**Conclusion.**
- **The threshold is set on all pairs:** at that common threshold, 1.7% of the giant-component negatives get an alarm, but 21% of the negatives in the rest.
- **So the low overall rate is carried by the many easy negatives of the giant component;** the hard cases sit in the rest.
- **Overfitting is small for the tree models:** the train vs validation gap is 0.23–0.26 pp. For logistic regression it is negative, i.e. no overfitting at all.

**Next step:** in the report, give the rate for the rest next to the overall rate. The error taxonomy should look at the rest separately.

## 7 · Light tuning for LightGBM and XGBoost

Two more settings per model from `GRID` (more trees and leaves / depth, or fewer and more regularised), on the same 5 splits. Chosen on validation pairs only.

**What the table shows:** same layout as the first table in section 6, now with all settings.

**How to read it:** pick the setting with the lowest FA rate at 99% recall only if it is better in most splits (see the paired table under it); otherwise keep the default.

In [ ]:
t0 = time.time()
res_tune, scores_tune = run([(m, k) for m in ("lightgbm", "xgboost") for k in range(1, len(GRID[m]))])
res_all = pd.concat([res, res_tune]); scores.update(scores_tune)
print(f"{time.time() - t0:.0f}s")
summary(res_all[res_all["model"].isin(["lightgbm", "xgboost"])])

**Conclusion.**
- **The larger setting 1 helps only marginally:** more trees and leaves/depth give 2.89% at 99% recall, against 2.91% / 2.93% for the default.
- **The smaller setting 2 is worse:** 3.07% / 3.11%.

**Next step:** see the paired table below.

In [ ]:
for s in SETS:
    print(f"feature set: {s}")
    display(paired(res_all[res_all["model"].isin(["lightgbm", "xgboost"])], s))

**Conclusion.**
- **Setting 1 beats the default in 4 of 5 splits** (LightGBM and XGBoost), but by only about 0.02–0.04 pp, at more than twice the training time.
- **Setting 2 loses in 5 of 5.**

**Next step:** keep the default for now. Repeat the tuning once the record features are in (more features can profit from deeper trees).

## 8 · Cost curve

For one model (`CHOSEN`), the number of false alarms per 1,000 screened pairs as a function of the required recall, from 90% to 99.9%. Computed on each of the 5 validation sets; the line is the mean, the band the range over splits.

Set `CHOSEN` from the tables above. The final curve for the report will be drawn on Vova's holdout, once, after the model is frozen.

In [ ]:
CHOSEN = ("name+record" if "name+record" in SETS else "name", "lightgbm", 0)
grid_r = np.round(np.arange(0.90, 0.9991, 0.001), 4)
curves = []
for rep, side in inner:
    va = dev[side == "test"]
    s = scores[(rep, *CHOSEN)]
    for r in grid_r:
        op = at_recall(va["y"], s, r)
        curves.append({"rep": rep, "recall": r, "fa_per_1000": op["false_alarms"] / len(va) * 1000,
                       "fa_rate": op["fa_rate"], "precision": 1 - op["false_alarms"] / op["alerts"]})
curves = pd.DataFrame(curves)
cc = curves.groupby("recall").agg(mean=("fa_per_1000", "mean"), lo=("fa_per_1000", "min"), hi=("fa_per_1000", "max"))

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(cc.index * 100, cc["lo"], cc["hi"], alpha=0.25, label="range over 5 splits")
ax.plot(cc.index * 100, cc["mean"], label="mean")
ax.axvline(TARGET * 100, ls="--", lw=1, color="grey")
ax.set_xlabel("recall (% of true matches found)"); ax.set_ylabel("false alarms per 1,000 pairs")
ax.set_title(f"Cost curve: {CHOSEN[1]} on {CHOSEN[0]} features (development pairs)"); ax.legend()
plt.show()

**Conclusion.** The curve is flat up to about 98% recall and rises steeply beyond 99%: every extra tenth of a percent of recall costs more false alarms than the one before.

**Next step:** the operating point at 99% recall is a reasonable default for the report; show 99.5% and 99.9% to make the cost of the last matches visible.

### Table: cost curve at selected points

**What it shows:** false alarms per 1,000 screened pairs, FA rate (% of negatives) and precision (share of alerts that are true matches), mean over the 5 splits.

**How to read it:** the step from 99% to 99.5% recall shows how expensive the last half percent of safety is.

In [ ]:
pts = curves[curves["recall"].isin([0.90, 0.95, 0.98, 0.99, 0.995, 0.999])]
t = pts.groupby("recall")[["fa_per_1000", "fa_rate", "precision"]].mean()
t["fa_rate"] *= 100; t["precision"] *= 100
t.rename(columns={"fa_per_1000": "false alarms per 1,000 pairs", "fa_rate": "FA rate %", "precision": "precision %"}).round(2)

**Conclusion.**
- **The last half-percent of recall is expensive:** from 99% to 99.5% recall, false alarms rise by more than half (9.1 → 14.0 per 1,000 pairs), and at 99.9% they are 67 per 1,000 (21.5% of the negatives).
- **Precision stays high** (98.7% at 99% recall) only because this data set is about 63% positive. Precision depends on the share of positives, the false-alarm rate does not.

**Next step:** state per-1,000 figures and precision only together with the positive share they were computed on, or rescale them to a realistic share.

## 9 · First look at error types

The false alarms of `CHOSEN` at 99% recall on validation split 0, tagged with simple draft rules. A pair can match several rules; its **main type** is the first rule that matches, in this order:

| Type | Rule (abbreviations: see notebook 04) |
| --- | --- |
| identical name | n_exact_name = 1 (same full name, word order ignored) |
| same company core, other legal form | company, n_sim_core ≥ 0.95, legal forms conflict or one-sided |
| common name | rarest shared full name carried by 6 or more entities (n_freq_name_band ≥ 2) or rarest shared word by more than 100 (n_freq_word_band ≥ 3) |
| initials only | n_initials = compatible (one side uses initials) |
| different script | n_script = different |
| near-identical spelling | n_sim_sort ≥ 0.9 |
| partial name | all words of the shorter name found in the longer one (n_token_coverage = 1) |
| other | none of the above |

These rules are a starting point for the error taxonomy, to be checked by reading the pairs.

In [ ]:
RULES = [
    ("identical name", lambda d: d["n_exact_name"] == 1),
    ("same company core, other legal form", lambda d: (d["is_person"] == 0) & (d["n_sim_core"] >= 0.95)
                                                      & d["n_legal_form"].isin(["conflict", "one_side"])),
    ("common name", lambda d: (d["n_freq_name_band"] >= 2) | (d["n_freq_word_band"] >= 3)),
    ("initials only", lambda d: d["n_initials"] == "compatible"),
    ("different script", lambda d: d["n_script"] == "different"),
    ("near-identical spelling", lambda d: d["n_sim_sort"] >= 0.9),
    ("partial name", lambda d: d["n_token_coverage"] == 1),
]
rep0, side0 = inner[0]
va0 = dev[side0 == "test"].copy()
va0["score"] = scores[(0, *CHOSEN)]
thr = at_recall(va0["y"], va0["score"], TARGET)["threshold"]
fa = va0[(va0["score"] >= thr) & (va0["y"] == 0)].copy()
fa["main_type"] = "other"
for name, rule in reversed(RULES):
    fa.loc[rule(fa).fillna(False).to_numpy(), "main_type"] = name
neg0 = va0[va0["y"] == 0]
print(f"split 0: {len(va0):,} validation pairs, {len(neg0):,} negatives, {len(fa):,} false alarms at {TARGET:.0%} recall")

### Table: false alarms by draft type

**What it shows:** number and share of false alarms per main type, split into persons and companies, and the share inside the giant component.

**How to read it:** the largest types are where the next features (Vova's birth date, country) or better name rules can help most. "identical name" cannot be fixed by any name feature.

In [ ]:
ft = fa.groupby("main_type").agg(false_alarms=("y", "size"), persons=("is_person", "sum"),
                                 in_giant=("group", lambda g: (g == "giant").mean()))
ft["companies"] = ft["false_alarms"] - ft["persons"]
ft["share"] = ft["false_alarms"] / len(fa)
ft = ft[["false_alarms", "share", "persons", "companies", "in_giant"]].sort_values("false_alarms", ascending=False)
ft.round(3)

**Conclusion.**
- **Identical names dominate:** 332 of 426 false alarms at 99% recall (78%), three quarters of them persons.
- **All other draft types are small:** partial name 7%, near-identical spelling 5%, different script 4.5%, common name 2%.
- **Different-script false alarms sit mostly in the giant component** (84%).

**Next step:** the error taxonomy should split "identical name" further, by whether record fields could separate the pair (notebook 08). The small types are only worth a closer look after the record features are in.

### Table: the hard ceiling for name features

**What it shows:** negative pairs whose full names are identical (n_exact_name = 1), in all labelled pairs and in the dev pairs, and how many of them get an alarm in split 0.

**How to read it:** these pairs look the same to every name feature, so at very high recall almost all of them become false alarms. They are the floor that only record features (birth date, country, ID numbers) can lower.

In [ ]:
ceil = pd.DataFrame({
    "negative pairs": [int((df["y"] == 0).sum()), int((dev["y"] == 0).sum()), len(neg0)],
    "with identical name": [int(((df["y"] == 0) & (df["n_exact_name"] == 1)).sum()),
                            int(((dev["y"] == 0) & (dev["n_exact_name"] == 1)).sum()),
                            int((neg0["n_exact_name"] == 1).sum())],
}, index=["all labelled pairs", "dev pairs", "validation split 0"])
ceil["share"] = ceil["with identical name"] / ceil["negative pairs"]
ceil.loc["validation split 0", "of these with an alarm"] = int((fa["n_exact_name"] == 1).sum())
ceil.round(3)

**Conclusion.**
- 3.7% of all negative pairs (6,034) have an identical name or alias on both sides.
- In split 0, 332 of these 537 get an alarm at 99% recall.
- With name features alone they cannot be separated: this is the floor of the name-only model.

**Next step:** measure how much the record features lower this floor (notebook 05: from 62% to 42% of these pairs with an alarm).

### Examples per type

**What it shows:** up to 3 false alarms per type with both names and the score, to check whether the draft rules make sense.

**How to read it:** if the examples of a type look alike, the rule is a good category; if not, it needs to be split or reworded.

In [ ]:
first = lambda v: (list(v)[0] if v is not None and len(v) else "")
ex = fa.sort_values("score", ascending=False).groupby("main_type").head(3)
ex = ex.assign(left=ex["left_name"].map(first), right=ex["right_name"].map(first))
ex[["main_type", "left", "right", "score", "group"]].sort_values("main_type").reset_index(drop=True)

**Conclusion.** The examples show three things:
- **Some "identical name" negatives match only through an alias** (the main names shown differ), e.g. Ukrainian and Russian company names that share a translated alias.
- **Some look like the same person** in another script or spelling (e.g. Evelina Bljodans), so they may be label errors.
- **Common company words** (bank, LLC in Russian) produce "common name" alarms.

**Next step:** notebook 08 checks the identical-name negatives against the record fields (birth date, country, ID number).

## 10 · Same person, several IDs: is that a problem for the split?

A positive pair means: two IDs, one real person or company. So one real entity can have several IDs, and a split by ID can put ID A of a person into training and ID B of the same person into validation.

Three checks:

1. **How common is it?** Link all IDs connected by positive pairs into one **cluster** (= one real entity, as far as the labels tell). Count clusters with more than one ID.
2. **How often does a real entity end up on both sides** of an inner split?
3. **Does it change the results?** Compare the model under a split **by cluster** (used everywhere else in this notebook) and a split **by ID**, so that no real entity is on both sides. If the false-alarm rates stay the same, splitting by ID is fine.

Clusters use the labels of all pairs, only to define who is the same entity. They are never a feature.

In [ ]:
parent_p = {}
def find_p(x):
    parent_p.setdefault(x, x)
    while parent_p[x] != x:
        parent_p[x] = parent_p[parent_p[x]]
        x = parent_p[x]
    return x
pos = df[df["y"] == 1]
for a, b in zip(pos["left_id"], pos["right_id"]):
    ra, rb = find_p(a), find_p(b)
    if ra != rb:
        parent_p[ra] = rb
ents_all = pd.unique(pd.concat([df["left_id"], df["right_id"]]))
cluster = pd.Series([find_p(e) for e in ents_all], index=ents_all)
df["cl_l"], df["cl_r"] = df["left_id"].map(cluster), df["right_id"].map(cluster)
csize = cluster.value_counts()
print(f"{len(ents_all):,} IDs in {len(csize):,} clusters; largest cluster: {csize.iloc[0]:,} IDs; "
      f"next: {csize.iloc[1:6].tolist()}")

### Table: clusters by number of IDs

**What it shows:** how many real entities (clusters) have 1, 2, 3–5, … IDs, how many IDs that is, and the share of all IDs.

**How to read it:** clusters with 2 or more IDs are the real entities that can end up on both sides of a split. If they are mostly small (2–5 IDs), a split by cluster is easy; one huge cluster would bring back the giant-component problem.

In [ ]:
bins = pd.cut(csize, [0, 1, 2, 5, 20, 100, 10**9], labels=["1", "2", "3–5", "6–20", "21–100", "> 100"])
ct = pd.DataFrame({"clusters": bins.value_counts(sort=False),
                   "IDs": csize.groupby(bins, observed=False).sum()})
ct["share of IDs"] = ct["IDs"] / ct["IDs"].sum()
ct.index.name = "IDs per cluster"
ct.round(3)

**Conclusion.**
- **Almost every ID belongs to a real entity with several IDs:** 95% of IDs are in clusters of 2 or more. 61% are in pairs of two IDs, the largest cluster has 36.
- **The clusters are small,** so a split by cluster is easy, unlike the split by connected component (giant component).

**Next step:** check how often a cluster ends up on both sides of the split by ID (next table).

### Table: real entities on both sides of the current split

**What it shows:** for a split by entity ID (salt `inner-0`, not the split of section 6): how many clusters with several IDs have IDs on both sides, and the share of the validation pairs where at least one entity has a "sibling" ID of the same cluster on the training side. Separately for positive and negative evaluation pairs.

**How to read it:** this is the exposure: how often the model has seen another ID of the same real entity during training.

In [ ]:
dev = df   # again, now with the cluster columns

def exposure(ent_is_test, eval_pairs, name):
    e = pd.DataFrame({"cl": cluster, "test": ent_is_test.reindex(cluster.index)}).dropna()
    per_cl = e.groupby("cl")["test"].agg(["min", "max", "size"])
    multi = per_cl[per_cl["size"] > 1]
    both = multi[(multi["min"] == 0) & (multi["max"] == 1)].index
    train_cl = set(e.loc[e["test"] == 0, "cl"])
    exp = eval_pairs["cl_l"].isin(train_cl) | eval_pairs["cl_r"].isin(train_cl)
    return {"split": name, "clusters with ≥2 IDs": len(multi), "of these on both sides": len(both),
            "eval pairs": len(eval_pairs), "exposed share (all)": exp.mean(),
            "exposed share (positive)": exp[eval_pairs["y"] == 1].mean(),
            "exposed share (negative)": exp[eval_pairs["y"] == 0].mean()}, exp

dev_ents = pd.unique(pd.concat([dev["left_id"], dev["right_id"]]))
ent_in0 = pd.Series(unit_hash(dev_ents, "inner-0") < 0.3, index=dev_ents).astype(int)
id_split0 = pair_side(dev, 0.3, "inner-0")   # split by entity ID, for this comparison only
va_0 = dev[id_split0 == "test"]
r_in0, exposed0 = exposure(ent_in0, va_0, "split by ID: validation vs train")
pd.DataFrame([r_in0]).set_index("split").T

**Conclusion.**
- **With the split by ID, a cluster on both sides is common:** 73,513 of 154,010 multi-ID clusters have IDs on both sides.
- **62% of the validation pairs contain an entity whose sibling ID was in training:** 57% of the positives and 71% of the negatives.

**Next step:** test whether that makes the results look better (experiment below).

### Experiment: split by ID vs split by cluster

Two schemes on the same pairs, the same models (XGBoost and LightGBM, default settings):

- **by cluster:** the 5 person-group splits of section 6 (a whole group goes to one side);
- **by ID:** 5 splits where every ID is assigned on its own (salts `inner-0` … `inner-4`), so IDs of one real entity can end up on both sides.

Note: a positive pair always lies inside one cluster, so it is never dropped; only negative pairs can be cross. The split by cluster therefore has a higher share of positives. The false-alarm rate (share of negatives) and recall do not depend on that share, so the two can be compared.

In [ ]:
def cluster_side(d, share, salt):
    cl = pd.unique(pd.concat([d["cl_l"], d["cl_r"]]))
    is_test = pd.Series(unit_hash(cl, salt) < share, index=cl)
    lt, rt = d["cl_l"].map(is_test).to_numpy(), d["cl_r"].map(is_test).to_numpy()
    return pd.Series(np.where(lt & rt, "test", np.where(~lt & ~rt, "train", "cross")), index=d.index)

dev = df   # now with cluster columns
schemes = {"by ID": [(rep, pair_side(dev, 0.3, f"inner-{rep}")) for rep in range(REPS)],
           "by cluster": inner}   # the person-group splits of section 6
cols = SETS["name"]
rows_c, sizes = [], []
t0 = time.time()
for scheme, splits in schemes.items():
    for rep, side in splits:
        tr, va = dev[side == "train"], dev[side == "test"]
        sizes.append({"scheme": scheme, "rep": rep, "train pairs": len(tr), "validation pairs": len(va),
                      "validation positive share": va["y"].mean(), "dropped pairs": int((side == "cross").sum())})
        for m in ("xgboost", "lightgbm"):
            s = make_model(m, GRID[m][0], tr[cols]).fit(tr[cols], tr["y"]).predict_proba(va[cols])[:, 1]
            if scheme == "by ID" and rep == 0 and m == "xgboost":
                s_id0 = s   # used in the exposed / not exposed table
            for _, r in by_group(va["y"], s, va["group"]).iterrows():
                rows_c.append({"scheme": scheme, "rep": rep, "model": m, "recall": r["recall"],
                               "group": r["group"], "fa_rate": r["fa_rate"]})
res_c = pd.DataFrame(rows_c)
print(f"{time.time() - t0:.0f}s")

### Table: size of the two kinds of split

**What it shows:** pairs in inner training and validation, the share of positives in validation and the pairs dropped, mean over the 5 splits.

**How to read it:** the split by cluster keeps all positive pairs, so its validation set has more positives and fewer pairs are dropped.

In [ ]:
pd.DataFrame(sizes).groupby("scheme").mean(numeric_only=True).drop(columns="rep").round(3)

**Conclusion.**
- **The split by cluster drops far fewer pairs:** 69,005 per split instead of 168,335. A positive pair always lies inside one cluster and is never dropped.
- **But it changes the mix:** validation is 85% positive instead of 69%.
- **False-alarm rate and recall are unaffected by that;** precision and per-pair figures are not.

**Next step:** compare false-alarm rates only (next table).

### Table: false alarms, split by ID vs split by cluster

**What it shows:** FA rate in %, mean ± standard deviation over the 5 splits, for all validation pairs and inside the giant component and the rest (99% recall for the groups).

**How to read it:** if seeing another ID of the same entity in training helped the model, the split by cluster would show clearly **more** false alarms. A difference within the standard deviation means the split by ID does not flatter the results.

In [ ]:
a = res_c[res_c["group"] == "all"].groupby(["model", "scheme", "recall"])["fa_rate"].agg(["mean", "std"]) * 100
a = a.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1).unstack("recall")
a.columns = [f"FA % at {c:.1%}" for c in a.columns]
g = (res_c[res_c["recall"] == TARGET].groupby(["model", "scheme", "group"])["fa_rate"].mean().unstack("group") * 100)
a.join(g[["giant", "rest"]].round(2).add_prefix("99%: "))

**Conclusion.**
- **The split by ID flatters the results:** the split by cluster gives more false alarms in both models. At 99% recall 3.28–3.29% instead of 2.91–2.93% (+0.4 pp, about +13%); at 99.5% 5.3% instead of 4.4–4.5%.
- **The difference comes mainly from the giant component** (1.7% → 2.0%); the rest barely changes.

**Next step:** use the split by cluster (positive-only groups) for all further work, as agreed with Vova.

### Table: exposed vs not exposed pairs (split by ID, inner split 0)

**What it shows:** the validation pairs of the first split by ID (XGBoost) divided into **exposed** (another ID of one of the two entities was in training) and **not exposed**. For each: number of pairs, FA rate among the negatives and recall among the positives, at the threshold for 99% recall on all validation pairs.

**How to read it:** a much better result for exposed pairs would point to a leak. Careful: the two groups also differ in other ways (exposed entities are the ones with many IDs, often on many lists), so this table is a hint, the experiment above is the test.

In [ ]:
s0 = s_id0
thr0 = at_recall(va_0["y"], s0, TARGET)["threshold"]
alarm0 = s0 >= thr0
rows_e = []
for name, m in [("exposed", exposed0.to_numpy()), ("not exposed", ~exposed0.to_numpy())]:
    y = va_0["y"].to_numpy()
    rows_e.append({"pairs": int(m.sum()), "negatives": int(((y == 0) & m).sum()),
                   "FA rate %": 100 * (alarm0 & (y == 0) & m).sum() / max(((y == 0) & m).sum(), 1),
                   "recall %": 100 * (alarm0 & (y == 1) & m).sum() / max(((y == 1) & m).sum(), 1)})
pd.DataFrame(rows_e, index=["exposed", "not exposed"]).round(2)

**Conclusion.**
- **Negatives whose entity had a sibling ID in training get fewer alarms:** 2.3% against 4.4% for the others.
- **The recall is about the same** (99.2% vs 98.8%).
- **Only a hint:** the two groups also differ in other ways, so the experiment above is the actual test.

**Next step:** none; both point the same way.

### Findings: section 10

- **Very common:** 95% of all IDs belong to a real entity with at least two IDs (2 IDs: 61% of IDs, 3–5: 17%, 6–20: 16%). The clusters are small: the largest has 36 IDs. A split by cluster is therefore easy, unlike the giant component (which also links negative pairs).
- **Often on both sides:** with the split by ID, 62% of the validation pairs (71% of the negatives) contain an entity whose "sibling" ID was in training.
- **It flatters the results a little.** Split by cluster (no known real entity on both sides) gives more false alarms: at 99% recall 3.28–3.29% instead of 2.91–2.93% (+0.4 pp, about +13%), at 99.5% 5.3% instead of 4.4–4.5%. The difference is about 2 standard deviations, in both models; it comes mainly from the giant component (1.7% → 2.0%), the rest barely changes (21.0% → 21.2–21.4%).
- **The exposed pairs are the easy ones:** within the split by ID, negatives whose entity had a sibling ID in training get an alarm in 2.3% of cases, the others in 4.4%. (A hint only: the two groups also differ in other ways.)
- **Why it can leak although no ID is a feature:** IDs of the same entity carry the same names. Pairs built from them have nearly identical feature values, so the model recognises rows it has practically seen before, for positives and for the typical confusions of that entity.
- **Fewer pairs lost, but only negatives:** split by cluster drops 69,005 pairs per split instead of 168,335, because positive pairs always lie inside one cluster. Validation is then 85% positive instead of 69%. FA rate and recall are not affected; precision and "false alarms per 1,000 pairs" are, so those must be rescaled to the real share of positives (62.6%).
- **Caveat:** "same entity" is known only through the labels. IDs of one person that were never paired, or a wrong negative label, are not caught.
- **Consequence (to agree with Vova):** split by person cluster for the holdout and the inner splits; report the difference to the split by ID.


## 11 · Scores for the error analysis (notebook 09)

The error analysis in notebook 09 needs the validation scores of the chosen model, so it does not have to train again. Saved here:

- **Model:** LightGBM, default setting (section 6).
- **Features:** `name+record` (15 name + 4 record features + `is_person`).
- **Split:** the 5 person-group splits of section 6, on Vova's train + val pairs (his test pairs are never loaded). His fixed `split` column itself is used only in the final run.

Two files in `data/processed/` (gitignored):
- `scores_dev.parquet`: one row per validation pair and split, with `pair_id`, `rep`, `y`, `group`, `score`.
- `splits_dev.parquet`: which side (train / test / cross) every pair was on in each split, so notebook 09 can retrain the same model for the SHAP analysis.

In [ ]:
FINAL_SET = "name+record" if "name+record" in SETS else "name"
final_cols = SETS[FINAL_SET]
score_parts, side_parts = [], []
t0 = time.time()
for rep, side in inner:
    tr, va = dev[side == "train"], dev[side == "test"]
    m = make_model("lightgbm", GRID["lightgbm"][0], tr[final_cols]).fit(tr[final_cols], tr["y"])
    score_parts.append(pd.DataFrame({"pair_id": va["pair_id"].to_numpy(), "rep": rep, "y": va["y"].to_numpy(),
                                     "group": va["group"].to_numpy(),
                                     "score": m.predict_proba(va[final_cols])[:, 1]}))
    side_parts.append(pd.DataFrame({"pair_id": dev["pair_id"].to_numpy(), "rep": rep, "side": side.to_numpy()}))
scores_dev = pd.concat(score_parts, ignore_index=True).assign(model="lightgbm", setting=0, feature_set=FINAL_SET,
                                                              split="by cluster")
scores_dev.to_parquet(C.DATA_PROCESSED / "scores_dev.parquet", index=False)
pd.concat(side_parts, ignore_index=True).to_parquet(C.DATA_PROCESSED / "splits_dev.parquet", index=False)
check = scores_dev.groupby("rep").apply(lambda g: at_recall(g["y"], g["score"], TARGET)["fa_rate"], include_groups=False)
print(f"saved {len(scores_dev):,} validation scores ({FINAL_SET}, {time.time() - t0:.0f}s); "
      f"FA rate at 99% recall per split: {', '.join(f'{x:.2%}' for x in check)}")

**Check:** the false-alarm rates printed above must match the LightGBM row in section 6 (same model, same splits).

## Findings (uncleaned data, name features only, all pairs, no holdout, 2026-11-18)

- **The three tree models are equal.** At 99% recall: LightGBM 2.91%, XGBoost 2.93%, HGB 2.95% false alarms (± 0.10–0.15). No model is better in more than 3 of 5 splits. Logistic regression needs 5.29% (worse in 5 of 5).
- **Tuning brings almost nothing:** the larger setting 1 (more trees, more leaves / depth) is better than the default in 4 of 5 splits, but only by about 0.02–0.04 pp at 99% recall; the smaller setting 2 is clearly worse. Repeat once the record features are in.
- **No overfitting:** train vs validation gap at 99% recall is 0.23–0.26 pp for the tree models.
- **Giant vs rest:** at the common threshold, 1.7% of the negatives in the giant component get an alarm, 21% of the negatives in the rest. The overall rate is low mainly because the giant component contributes many easy negatives.
- **Cost curve (LightGBM):** 9.1 false alarms per 1,000 pairs at 99% recall, 14.0 at 99.5%, 67.2 at 99.9% (21.5% of the negatives). Per-1,000 numbers depend on the share of positives in the validation set.
- **78% of the false alarms at 99% recall are identical names** (332 of 426 in split 0). 3.7% of all negative pairs (6,034) have the same full name on both sides (word order and transliteration ignored); 332 of the 537 in split 0 get an alarm. With name features alone this is the floor; only record features (birth date, country, ID numbers) can lower it.
- **Check the labels:** some negatives with identical names look like the same person (examples above). Notebook 08 checks them against the record fields.
- **Other draft error types are small:** partial name 7%, near-identical spelling 5%, different script 4.5%, common name 2%.
- **The split by ID is a little optimistic** (section 10): +0.4 pp false alarms at 99% recall when no real entity is on both sides.


## Moving this into code later

When the notebook is settled, the cells move as follows; the logic stays the same.

| Notebook section | Goes to |
| --- | --- |
| 1 load + merge | `src/screening/dataset.py`, `scripts/03_features.py` |
| 2–3 split rule, giant component | `src/screening/split.py` (settings in `config.py`) |
| 4 evaluation helpers | `src/screening/evaluate.py` |
| 5 models + grid | `src/screening/models.py` (grid in `config.py`) |
| 6–7 comparison, tuning | `scripts/04_train.py` → `reports/02-model.md` |
| final run on Vova's holdout | `scripts/04_train.py --final` (once, at the end) |
| 8 cost curve | `scripts/05_threshold.py` → `reports/03-cost-curve.md` |
| 9 error types | `scripts/06_errors.py` → `reports/04-error-taxonomy.md` |
| 11 saved scores | written by `scripts/04_train.py`, read by notebook 09 / `scripts/06_errors.py` |